In [ ]:
import pandas as pd
from src.data.loaders import (load_amzn_books, split_and_reindex)

In [ ]:
config = {
    "amzn-books": {
        "max_seq_len": 50,
        "test_quantile": 0.1,
        "interactions_path": "/kaggle/input/datasets/andrewkhoroshilov/datasets-action-speak-louder-then-words/ratings_Books.csv/Books.csv", # https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Books.csv.gz
        "col_mapping": {
            "user_id": "user_id",
            "parent_asin": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
}

In [ ]:
amzn_df = load_amzn_books(config["amzn-books"]["interactions_path"], config=config["amzn-books"])

In [ ]:
amzn_train, amzn_test, amzn_data_description = split_and_reindex(amzn_df, config=config["amzn-books"])

In [ ]:
datasets = {
    "amzn-books": {
        "train": amzn_train,
        "test": amzn_test,
        "desc": amzn_data_description,
    },
}

In [24]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,amzn-books,10931769,512200,11443969,0.955,0.045,1160538,215463,900095


In [25]:
import torch

from src.training.hstu import (
    HSTUExperimentConfig,
    build_hstu_dataloaders,
    evaluate_hstu,
    train_hstu,
)
from src.models.hstu import HSTUModel


# HSTU Amazon reviews

In [26]:
hstu_config = HSTUExperimentConfig(
    max_seq_len=config["amzn-books"]["max_seq_len"],
    train_batch_size=128,
    eval_batch_size=128,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=0.0,
    topk=100,
    embedding_dim=64,
    num_blocks=4,
    num_heads=4,
    linear_dim=16,
    attention_dim=16,
    input_dropout_rate=0.5,
    linear_dropout_rate=0.5,
    num_negatives=512,
    sampling_strategy="local",
    softmax_temperature=0.05,
    user_embedding_norm="l2_norm",
    enable_relative_attention_bias = True,
    l2_norm_embeddings=True,
    l2_norm_eps=1e-6,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

hstu_train_loader, hstu_eval_loader, hstu_train_dataset, hstu_eval_dataset, hstu_targets = build_hstu_dataloaders(
    train=amzn_train,
    test=amzn_test,
    max_seq_len=hstu_config.max_seq_len,
    train_batch_size=hstu_config.train_batch_size,
    eval_batch_size=hstu_config.eval_batch_size,
    user_col=amzn_data_description["users"],
    item_col=amzn_data_description["items"],
    time_col=amzn_data_description["timestamp"],
    num_workers=0,
)


len(hstu_train_dataset), len(hstu_eval_dataset), len(hstu_targets)

(1161643, 215463, 215463)

In [27]:
hstu_model = HSTUModel(
    num_items=amzn_data_description["n_items"],
    embedding_dim=hstu_config.embedding_dim,
    max_seq_len=hstu_config.max_seq_len,
    num_blocks=hstu_config.num_blocks,
    num_heads=hstu_config.num_heads,
    linear_dim=hstu_config.linear_dim,
    attention_dim=hstu_config.attention_dim,
    num_negatives=hstu_config.num_negatives,
    softmax_temperature=hstu_config.softmax_temperature,
    sampling_strategy=hstu_config.sampling_strategy,
    user_embedding_norm=hstu_config.user_embedding_norm,
    l2_norm_embeddings=hstu_config.l2_norm_embeddings,
    l2_norm_eps=hstu_config.l2_norm_eps,
    item_id_offset=hstu_config.item_id_offset,
    input_dropout_rate=hstu_config.input_dropout_rate,
    linear_dropout_rate=hstu_config.linear_dropout_rate,
    attn_dropout_rate=hstu_config.attn_dropout_rate,
)

hstu_model.to(hstu_config.device)
hstu_optimizer = torch.optim.AdamW(
    hstu_model.parameters(),
    lr=hstu_config.learning_rate,
    weight_decay=hstu_config.weight_decay,
)


In [28]:
params_sum = 0
trainable_sum = 0

for name, module in hstu_model.encoder.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{name}: total={params:,}, trainable={trainable:,}")

    params_sum += params
    trainable_sum += trainable
print()
print(f"params_sum={params_sum:,}, trainable_sum={trainable_sum:,}")

item_embeddings: total=57,606,144, trainable=57,606,144
pos_embeddings: total=3,200, trainable=3,200
input_dropout: total=0, trainable=0
blocks: total=84,112, trainable=84,112

params_sum=57,693,456, trainable_sum=57,693,456


In [29]:
hstu_losses, eval_history = train_hstu(
    model=hstu_model,
    train_loader=hstu_train_loader,
    optimizer=hstu_optimizer,
    num_epochs=hstu_config.num_epochs,
    device=hstu_config.device,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=amzn_data_description["n_items"],
    topk=hstu_config.topk,
    filter_seen=hstu_config.filter_seen,
)
hstu_losses

epoch 1/10:   0%|          | 0/9075 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/1684 [00:00<?, ?it/s]

epoch 1/10: loss=5.3978, coverage=0.0126, hitrate=0.0500, ndcg=0.0072, recall=0.0241


epoch 2/10:   0%|          | 0/9075 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/1684 [00:00<?, ?it/s]

...

epoch 9/10: loss=3.0766, coverage=0.2261, hitrate=0.1107, ndcg=0.0199, recall=0.0594


epoch 10/10:   0%|          | 0/9075 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/1684 [00:00<?, ?it/s]

epoch 10/10: loss=3.0249, coverage=0.2354, hitrate=0.1111, ndcg=0.0200, recall=0.0599


[5.3977920508581745,
 4.406066654063453,
 3.92561436597966,
 3.635945079871774,
 3.452518914724513,
 3.320718136989709,
 3.2203504328294232,
 3.141041239641258,
 3.0766359779263333,
 3.024943404394733]

In [30]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=amzn_data_description["n_items"],
    topk=10,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/1684 [00:00<?, ?it/s]

{'hitrate': 0.031142237878429242,
 'recall': 0.014341736131323966,
 'ndcg': 0.009110060031951027,
 'coverage': 0.08990273248934835}

In [31]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=amzn_data_description["n_items"],
    topk=50,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/1684 [00:00<?, ?it/s]

{'hitrate': 0.07946608002302019,
 'recall': 0.04033939981099343,
 'ndcg': 0.01595263741271996,
 'coverage': 0.18081758036651688}

In [32]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=amzn_data_description["n_items"],
    topk=100,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/1684 [00:00<?, ?it/s]

{'hitrate': 0.11114205223170567,
 'recall': 0.05989055509904843,
 'ndcg': 0.019951620245949386,
 'coverage': 0.23542737155522472}

In [33]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=amzn_data_description["n_items"],
    topk=200,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/1684 [00:00<?, ?it/s]

{'hitrate': 0.14913001304168233,
 'recall': 0.08569106753340024,
 'ndcg': 0.024506540817905556,
 'coverage': 0.29949283131225035}

In [34]:
checkpoint = {
    "epoch": hstu_config.num_epochs,
    "model_state_dict": hstu_model.state_dict(),
    "optimizer_state_dict": hstu_optimizer.state_dict(),
    "loss": hstu_losses,
}

torch.save(checkpoint, "checkpoint.pt")